In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
import joblib
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg


# ================= LOAD MODELS =================
try:
    best_model = joblib.load("best_ml_model.joblib")
    scaler = joblib.load("feature_scaler.joblib")
    feature_extractor = joblib.load("feature_extractor.joblib")
    preprocessor = joblib.load("image_preprocessor.joblib")
    test_data = joblib.load("test_data.joblib")
    model_results = joblib.load("model_results.joblib")
except Exception as e:
    raise RuntimeError(f"Error loading models: {e}")

CLASS_NAMES = test_data["classes"]

# ================= MAIN WINDOW =================
root = tk.Tk()
root.title("Brain Tumor Classification System")
root.geometry("900x600")

class ImagePreprocessor:
    def __init__(self, target_size=(224, 224)):
        self.target_size = target_size

    def preprocess_image(self, image_path):
        if not os.path.exists(image_path):
            return None

        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None

        img = cv2.resize(img, self.target_size)
        img = img / 255.0
        return img

# ================= FUNCTIONS =================
def load_image():
    global img_path, img_tk

    img_path = filedialog.askopenfilename(
        filetypes=[("Image files", "*.jpg *.jpeg *.png *.tif")]
    )
    if not img_path:
        return

    image = Image.open(img_path).resize((250, 250))
    img_tk = ImageTk.PhotoImage(image)
    image_label.config(image=img_tk)
    image_label.image = img_tk

    result_label.config(text="")
    confidence_label.config(text="")
    clear_plot()


def predict_image():
    if not img_path:
        messagebox.showwarning("Warning", "Please upload an image first.")
        return

    try:
        img = cv2.imread(img_path)
        temp_path = "temp_image.jpg"
        cv2.imwrite(temp_path, img)

        preprocessed_img = preprocessor.preprocess_image(temp_path)
        os.remove(temp_path)

        features = feature_extractor.extract_all_features(preprocessed_img)
        features_array = np.array(list(features.values())).reshape(1, -1)
        scaled_features = scaler.transform(features_array)

        prediction = best_model.predict(scaled_features)[0]
        probabilities = best_model.predict_proba(scaled_features)[0]

        predicted_class = CLASS_NAMES[prediction]
        confidence = probabilities[prediction]

        result_label.config(text=f"Prediction: {predicted_class}")
        confidence_label.config(text=f"Confidence: {confidence:.2%}")

        plot_probabilities(probabilities)

    except Exception as e:
        messagebox.showerror("Error", str(e))


def plot_probabilities(probs):
    clear_plot()

    fig, ax = plt.subplots(figsize=(4, 3))
    ax.bar(CLASS_NAMES, probs)
    ax.set_ylabel("Probability")
    ax.set_title("Class Probabilities")
    ax.set_ylim(0, 1)

    canvas = FigureCanvasTkAgg(fig, master=plot_frame)
    canvas.draw()
    canvas.get_tk_widget().pack()


def clear_plot():
    for widget in plot_frame.winfo_children():
        widget.destroy()


# ================= UI LAYOUT =================
title = tk.Label(root, text="🧠 Brain Tumor Classifier", font=("Arial", 20, "bold"))
title.pack(pady=10)

main_frame = tk.Frame(root)
main_frame.pack(fill="both", expand=True)

# Left panel
left_frame = tk.Frame(main_frame)
left_frame.pack(side="left", padx=20)

image_label = tk.Label(left_frame)
image_label.pack(pady=10)

upload_btn = tk.Button(left_frame, text="Upload MRI Image", command=load_image)
upload_btn.pack(pady=5)

predict_btn = tk.Button(left_frame, text="Predict", command=predict_image)
predict_btn.pack(pady=5)

# Right panel
right_frame = tk.Frame(main_frame)
right_frame.pack(side="right", padx=20, fill="both", expand=True)

result_label = tk.Label(right_frame, text="", font=("Arial", 14, "bold"))
result_label.pack(pady=10)

confidence_label = tk.Label(right_frame, text="", font=("Arial", 12))
confidence_label.pack(pady=5)

plot_frame = tk.Frame(right_frame)
plot_frame.pack(fill="both", expand=True)

# ================= START APP =================
img_path = None
root.mainloop()
